In [4]:
import os

project_path = r"C:\Users\HP ZBOOK\OneDrive\Desktop\EE4216_Biscuit_Detection"

os.chdir(project_path)

print(os.getcwd())
print(os.listdir())

C:\Users\HP ZBOOK\OneDrive\Desktop\EE4216_Biscuit_Detection
['input_images', 'output_images']


In [10]:
input_folder = "input_images"
output_folder = "output_images"

print(os.listdir(input_folder))

['image1.jpeg', 'image2.jpeg', 'image3.jpeg', 'image4.jpeg', 'image5.jpeg', 'image6.jpeg', 'image7.jpeg', 'image8.jpeg']


In [11]:
import cv2
import numpy as np
import os

input_folder = "input_images"
output_folder = "output_images"
os.makedirs(output_folder, exist_ok=True)

image_files = [f for f in os.listdir(input_folder)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for file_name in image_files:

    image_path = os.path.join(input_folder, file_name)
    image = cv2.imread(image_path)

    if image is None:
        continue

    output = image.copy()

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower = np.array([5, 30, 30])
    upper = np.array([45, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    intact = 0
    broken = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 3000:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        circularity = 4 * np.pi * area / (perimeter * perimeter)
        aspect_ratio = w / float(h)

        # ROUND biscuit rule
        if circularity > 0.78 and 0.80 < aspect_ratio < 1.20:
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1
        else:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken += 1

        cv2.drawContours(output, [cnt], -1, color, 2)
        cv2.rectangle(output, (x, y), (x+w, y+h), color, 2)
        cv2.putText(output, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.putText(output, f"Intact: {intact}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
    cv2.putText(output, f"Broken: {broken}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    cv2.imwrite(os.path.join(output_folder, "result_" + file_name), output)
    print(file_name, "Intact:", intact, "Broken:", broken)
    

image1.jpeg Intact: 2 Broken: 2
image2.jpeg Intact: 2 Broken: 3
image3.jpeg Intact: 2 Broken: 2
image4.jpeg Intact: 2 Broken: 3
image5.jpeg Intact: 2 Broken: 3
image6.jpeg Intact: 2 Broken: 3
image7.jpeg Intact: 2 Broken: 2
image8.jpeg Intact: 2 Broken: 2


In [12]:
readme_text = """# Broken Biscuit Detection using Classical Image Processing

## Problem Description
This project detects biscuits and classifies them as intact or broken using classical image processing techniques.

## Tools and Libraries
- Python
- OpenCV
- NumPy
- Matplotlib
- Jupyter Notebook

## Methods Used
- HSV color conversion
- Thresholding
- Morphological operations
- Contour detection
- Shape analysis

## How to Run
1. Open Anaconda Prompt
2. Activate environment:
   conda activate ee4216
3. Run:
   jupyter notebook

## Output
- Intact Biscuit
- Broken Biscuit
"""

with open("README.md", "w") as f:
    f.write(readme_text)

print("README.md created")

README.md created
